<a href="https://colab.research.google.com/github/RonakkudalAI/Deep-Neural-Network/blob/main/03_hr_question_answering_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question Answering System using Hugging Face

## HR case study: answering employee policy questions

Employees often ask repeated HR questions:

- How many paid leaves do I get?
- What is the notice period?
- Can I work from home?
- What is the medical insurance coverage?

Instead of asking HR for every basic question, employees can ask a question and receive an answer from the policy text.

In this notebook, we will build an end-to-end extractive question answering system.


## Learning roadmap

1. Understand the HR support problem.
2. Create a small HR policy knowledge base.
3. Understand question answering.
4. Learn extractive versus generative QA.
5. Retrieve the most relevant policy section.
6. Use a Hugging Face QA model.
7. Return an answer with confidence score.
8. Test realistic employee questions.
9. Create business-friendly answer output.
10. Discuss limitations and alternatives.


## Code microscope: important question answering words

| Term | Simple meaning | Why it matters |
|---|---|---|
| `TfidfVectorizer` | Converts text into numbers based on important words. | We use it to search for the most relevant HR policy section. |
| `policy_vectors` | Numeric representation of all policy sections. | This lets us compare a question with every policy section. |
| `cosine_similarity` | A score showing how similar two pieces of text are. | Higher score means the question probably matches that section better. |
| `pipeline("question-answering")` | A ready-to-use Hugging Face QA workflow. | It loads tokenizer, model, preprocessing, prediction, and output formatting. |
| `question` | What the employee asks. | The model uses it to know what answer to look for. |
| `context` | The policy text given to the model. | Extractive QA can only answer from this text. |
| `score` | Model confidence for the extracted answer. | Low confidence should trigger review. |
| `start` and `end` | Character positions of the answer in the context. | They show where the answer came from. |


## 1. Why question answering matters in HR

HR teams receive many repeated questions. Many answers already exist in policy documents, but employees may not know where to look.

A question answering system helps by:

- Reducing repeated manual work for HR.
- Giving employees faster answers.
- Making policy documents easier to use.
- Keeping answers grounded in approved text.


## 2. Simple explanation of question answering

Question answering means giving the model:

1. A question.
2. A context passage that contains the answer.

The model then finds the answer inside the context.

Real-world analogy:

Imagine giving a student one paragraph from an HR handbook and asking, "What is the notice period?" The student reads the paragraph and highlights the answer. An extractive QA model does something similar.


## 3. Tiny example

Context:

"Employees are eligible for 18 paid leaves per calendar year. Unused paid leaves cannot be carried forward."

Question:

"How many paid leaves are employees eligible for?"

Answer:

"18 paid leaves per calendar year"


## 4. Extractive QA versus generative QA

### Extractive QA

The answer is copied from the provided context.

Good for:

- HR policy answers
- Legal or compliance documents
- Product manuals
- FAQ pages

### Generative QA

The model writes a new answer in its own words.

Good for:

- Conversational assistants
- Explanations
- Creative responses

For HR policies, extractive QA is safer because the answer stays close to the approved policy text.


In [ ]:
# This cell installs the libraries used in this notebook.
# Run it once if you are using a fresh environment.
%pip install -q transformers scikit-learn pandas numpy torch


In [ ]:
# pandas helps us organize policy sections and results in tables.
import pandas as pd

# numpy helps with numerical operations.
import numpy as np

# TfidfVectorizer converts policy text into searchable numeric vectors.
from sklearn.feature_extraction.text import TfidfVectorizer

# cosine_similarity measures how similar a question is to each policy section.
from sklearn.metrics.pairwise import cosine_similarity

# pipeline is the easiest way to use pretrained Hugging Face models.
from transformers import pipeline


## 5. Create an HR policy knowledge base

In a real company, this content may come from:

- HR policy PDFs
- Employee handbook
- Internal wiki
- Benefits documentation
- Leave policy documents

For this classroom case study, we will create a small policy knowledge base directly inside the notebook.


In [ ]:
# We create a list of HR policy sections.
# Each section has a topic and approved policy text.
policy_sections = [
    {
        "topic": "Paid Leave",
        "policy_text": "Employees are eligible for 18 paid leaves per calendar year. Paid leaves must be approved by the reporting manager before the leave date. Unused paid leaves cannot be carried forward to the next calendar year.",
    },
    {
        "topic": "Sick Leave",
        "policy_text": "Employees are eligible for 10 sick leaves per calendar year. If sick leave exceeds two consecutive working days, the employee must submit a medical certificate to the HR team.",
    },
    {
        "topic": "Work From Home",
        "policy_text": "Employees may work from home up to two days per week with manager approval. Work from home is not allowed during mandatory office events, client workshops, or team training days.",
    },
    {
        "topic": "Notice Period",
        "policy_text": "The standard notice period for full-time employees is 60 calendar days. The notice period starts from the date the resignation email is formally acknowledged by HR.",
    },
    {
        "topic": "Medical Insurance",
        "policy_text": "All full-time employees are covered under the company medical insurance plan from their date of joining. The plan covers the employee, spouse, and up to two dependent children.",
    },
    {
        "topic": "Learning Reimbursement",
        "policy_text": "Employees can claim learning reimbursement up to 25000 rupees per financial year for approved courses, certifications, and professional workshops related to their role.",
    },
    {
        "topic": "Probation",
        "policy_text": "New employees have a probation period of six months. Confirmation depends on performance review, manager feedback, attendance, and completion of mandatory onboarding courses.",
    },
    {
        "topic": "Holiday Policy",
        "policy_text": "The company provides 12 public holidays every calendar year. The holiday calendar is published by HR before the start of the year and may vary by office location.",
    },
]

# We convert the policy sections into a DataFrame.
policy_df = pd.DataFrame(policy_sections)

# We display the knowledge base.
policy_df


## 6. Why retrieval is needed before QA

A QA model needs a context passage.

If we have many policy sections, we first need to find the section most likely to contain the answer.

This gives us a simple two-step system:

1. Retrieval: find the best policy section.
2. Question answering: extract the answer from that section.

This pattern is common in real-world document QA systems.


In [ ]:
# TfidfVectorizer converts text into numeric vectors.
# TF means term frequency: how often a word appears in a document.
# IDF means inverse document frequency: words that appear everywhere become less important.
# stop_words="english" removes very common English words such as "the", "is", and "and".
vectorizer = TfidfVectorizer(stop_words="english")

# fit_transform does two things:
# 1. fit learns the vocabulary from all policy sections.
# 2. transform converts each policy section into a numeric vector.
policy_vectors = vectorizer.fit_transform(policy_df["policy_text"])

# The shape tells us:
# - number of rows: number of policy sections.
# - number of columns: number of unique useful words learned by TF-IDF.
policy_vectors.shape


In [ ]:
# This function retrieves the policy section most related to an employee question.
def retrieve_policy_section(question):
    # question is the employee's natural-language question.
    # Example: "How many days is the notice period?"

    # transform converts the question into the same TF-IDF vector format as the policy sections.
    # We use transform, not fit_transform, because the vocabulary was already learned from policies.
    question_vector = vectorizer.transform([question])

    # cosine_similarity compares the question vector with every policy vector.
    # The result is one similarity score per policy section.
    similarity_scores = cosine_similarity(question_vector, policy_vectors)[0]

    # np.argmax returns the index of the highest similarity score.
    # This is our best guess for the policy section containing the answer.
    best_index = int(np.argmax(similarity_scores))

    # We use the best index to fetch the policy topic.
    best_topic = policy_df.loc[best_index, "topic"]

    # We use the same index to fetch the policy text.
    best_context = policy_df.loc[best_index, "policy_text"]

    # We convert the best score into a normal Python float for easy display.
    best_score = float(similarity_scores[best_index])

    # Return the selected topic, selected text, and retrieval score.
    return best_topic, best_context, best_score


In [ ]:
# We test retrieval with a sample employee question.
sample_question = "How many days do I need to serve after resignation?"

# We retrieve the best policy section.
topic, context, score = retrieve_policy_section(sample_question)

# We print the selected topic.
print("Selected policy topic:", topic)

# We print the retrieval similarity score.
print("Retrieval score:", score)

# We print the selected context.
print("Selected context:")
print(context)


## 7. Load a pretrained QA model

We will use a Hugging Face question-answering pipeline.

The model used here is trained to find answer spans in a given context.

Important:

This model does not search the internet. It only answers from the context we provide.


In [ ]:
# pipeline is Hugging Face's beginner-friendly interface.
# task="question-answering" tells Hugging Face we want an extractive QA system.
# Extractive means the answer is copied from the context text.
qa_pipeline = pipeline(
    # The task controls the type of NLP workflow.
    task="question-answering",

    # This checkpoint is a DistilBERT model fine-tuned on SQuAD, a question-answering dataset.
    # It has learned to find answer spans inside a paragraph.
    model="distilbert-base-cased-distilled-squad",
)


## 8. Ask one question

The QA pipeline expects:

- `question`: what the employee asks.
- `context`: the policy text where the answer should be found.

The output includes:

- `answer`: extracted answer text.
- `score`: model confidence.
- `start` and `end`: character positions inside the context.


In [ ]:
# We define one employee question.
question = "How many paid leaves do employees get?"

# First, we retrieve the policy section that is most likely to contain the answer.
topic, context, retrieval_score = retrieve_policy_section(question)

# qa_pipeline receives two inputs:
# - question: what the employee wants to know.
# - context: the policy text where the answer should be found.
answer_result = qa_pipeline(
    question=question,
    context=context,
)

# answer_result is a dictionary.
# Important keys are:
# - answer: extracted text span.
# - score: confidence score.
# - start: starting character position of the answer.
# - end: ending character position of the answer.

# We print the topic selected by retrieval.
print("Policy topic:", topic)

# We print the context used by the QA model.
print("\nContext:")
print(context)

# We print the answer extracted by the model.
print("\nAnswer:")
print(answer_result["answer"])

# We print model confidence.
print("\nQA confidence score:")
print(answer_result["score"])


## 9. Build a reusable HR question answering function

Now we combine retrieval and QA into one function.

The function will:

1. Receive an employee question.
2. Find the most relevant policy section.
3. Extract the answer from that section.
4. Return a business-friendly result.


In [ ]:
# This function combines retrieval and question answering.
def answer_hr_question(question):
    # Step 1: find the most relevant policy section for the question.
    # topic is the section name, such as "Paid Leave".
    # context is the policy paragraph that will be passed to the QA model.
    # retrieval_score tells us how strongly the question matched that section.
    topic, context, retrieval_score = retrieve_policy_section(question)

    # Step 2: ask the extractive QA model to find the answer inside the context.
    answer_result = qa_pipeline(
        question=question,
        context=context,
    )

    # Step 3: create a business-friendly result dictionary.
    result = {
        # Store the original employee question.
        "question": question,

        # Store the policy section selected by retrieval.
        "policy_topic": topic,

        # Store the exact answer span extracted by the model.
        "answer": answer_result["answer"],

        # Store the QA confidence score as a normal Python float.
        "qa_confidence": float(answer_result["score"]),

        # Store retrieval score so we can judge whether the selected policy was reliable.
        "retrieval_score": retrieval_score,

        # Store the source policy text for transparency.
        "source_context": context,
    }

    # Return the final answer package.
    return result


In [ ]:
# We create realistic employee questions.
employee_questions = [
    "How many sick leaves are available in a year?",
    "Who is covered under medical insurance?",
    "How many days is the notice period?",
    "Can I work from home three days every week?",
    "How much learning reimbursement can I claim?",
]

# We create an empty list to store answers.
answers = []

# We answer each employee question.
for question in employee_questions:
    # We call the reusable QA function.
    result = answer_hr_question(question)

    # We append the result.
    answers.append(result)

# We convert answers into a DataFrame.
answers_df = pd.DataFrame(answers)

# We display selected columns for easy reading.
answers_df[["question", "policy_topic", "answer", "qa_confidence", "retrieval_score"]]


## 10. Output explanation

For each question:

- `policy_topic` shows the policy section selected by retrieval.
- `answer` shows the text extracted by the model.
- `qa_confidence` shows how confident the QA model is about the extracted answer.
- `retrieval_score` shows how strongly the question matched the selected policy section.

Low confidence or low retrieval score should trigger human review.


In [ ]:
# This function adds a review recommendation.
def review_recommendation(qa_confidence, retrieval_score):
    # Very low retrieval means we may have selected the wrong policy section.
    if retrieval_score < 0.10:
        return "Ask HR to review because the relevant policy section is unclear"

    # Low QA confidence means the model may not have found a reliable answer.
    if qa_confidence < 0.20:
        return "Ask HR to review because model confidence is low"

    # Otherwise the answer can be shown with the policy source.
    return "Show answer with policy source"

# We apply the review recommendation to each answer.
answers_df["recommendation"] = answers_df.apply(
    lambda row: review_recommendation(row["qa_confidence"], row["retrieval_score"]),
    axis=1,
)

# We display the business-ready result.
answers_df[["question", "answer", "policy_topic", "recommendation"]]


## 11. Business interpretation

This system can become an HR self-service assistant.

A good production workflow would:

- Show the answer.
- Show the policy source.
- Show a confidence or review flag.
- Allow the employee to contact HR if the answer is unclear.
- Log unanswered questions so HR can improve documentation.


## 12. Limitations and risks

Important limitations:

- Extractive QA can only answer from the provided context.
- If retrieval selects the wrong policy section, the final answer may be wrong.
- Some employee questions need judgment, not just text extraction.
- Policy text must be updated when HR rules change.
- Sensitive employee data must be protected.

For HR, the system should assist humans, not replace policy owners.


## 13. Alternatives

Other options:

- Manual FAQ search.
- Keyword search using company intranet.
- Semantic search using sentence embeddings.
- Generative QA using large language models.
- Retrieval-augmented generation with citations.

For freshers, this notebook uses TF-IDF retrieval plus extractive QA because it is simple, explainable, and grounded in source text.


## 14. Student practice

Try these tasks:

1. Add a policy section for maternity or paternity leave.
2. Ask five new questions in your own words.
3. Test a question that is not covered by any policy.
4. Change the confidence threshold.
5. Print the source context below every answer.

## Final takeaway

You built an end-to-end HR question answering system that retrieves a policy section and extracts an answer using a Hugging Face QA model.
